# Task 2 Plan: State Transition Dynamics

## Objective
Define action-conditioned transition equations for a stable and interpretable RL environment.

## Plan
1. Formalize transition variables and helper indicators.
2. Define per-action updates for AskInfo, ProvideSolution, AffectiveRepair, Escalate, Close.
3. Define derived success model p_t and success sampling.
4. Define failure acceleration and frustration dynamics.
5. Define termination logic and edge-case guards.

## State Used by Dynamics
Static: tier, customer_value_weight, problem_type, difficulty d, persona (rho, sigma, tau).
Dynamic: frustration f_t, information i_t, turn n_t, resolved r_t, failed_streak k_t.
Derived: p_t = g(state).

## Core Derived Success Function
Use a bounded logistic model:

$$
p_t = igma\left(\beta_0 + \beta_i i_t - \beta_d d - \beta_f f_t + \beta_a u(a_t) + \beta_p v(\text{problemtype})\right)
$$

where sigma is sigmoid, and u(a_t) is an action quality offset.

## Transition Helper Terms
- fail_t: 1 if a ProvideSolution attempt fails, else 0
- progress_t: bounded proxy for forward motion (for example, increase in information or successful repair)
- repair_t: 1 if AffectiveRepair is effective, else 0

Frustration update skeleton:

$$
\Delta f_t = a_0(1-rho) + a_1 sigma \cdot fail_t + a_2 \cdot \phi(k_t, tau) - a_3 \cdot repair_t - a_4 \cdot progress_t + \epsilon_f
$$

$$
f_{t+1} = clip(f_t + \Delta f_t, 0, 1)
$$

with\n

$$
\phi(k_t, tau) = \frac{k_t}{k_t + c}(1-tau)
$$

## Per-Action Transition Rules

### 1) AskInfo
- information gain: i_{t+1} = clip(i_t + delta_info + eps_i, 0, 1)
- failed_streak reset: k_{t+1} = k_t (or reset to 0 if using strict failure-only memory)
- frustration: mild increase from extra turn unless strong progress

### 2) ProvideSolution
- compute p_t from current state
- sample success_t ~ Bernoulli(p_t)
- if success_t = 1: r_{t+1} = 1, k_{t+1} = 0, frustration drops
- if success_t = 0: r_{t+1} = 0, k_{t+1} = k_t + 1, frustration increases via failure shock

### 3) AffectiveRepair
- little direct info gain, but frustration relief if repair is effective
- k_{t+1} typically unchanged
- optional small p_t uplift through reduced frustration

### 4) Escalate
- terminal action
- r_{t+1} set by design choice (usually unresolved in automated channel, then terminal)

### 5) Close
- if confidence threshold met (for example p_t >= theta_close and low frustration): success terminal
- else close-failure terminal

## Turn Counter
n_{t+1} = n_t + 1 for nonterminal transitions.

## Termination Conditions
Episode ends if any is true:
1. resolved r_{t+1} = 1
2. action is Escalate
3. action is Close
4. n_{t+1} >= T_max

## Stability Guards
- clip all continuous states to [0,1]
- keep small bounded noise for realism, not chaos
- enforce monotonicity where intended (for example, info should not systematically decrease)
- ensure p_t is recomputed each step and never separately drifted

## Deliverables for Task 2
- final transition equations per action
- success and failure sampling protocol
- terminal-state and edge-case handling rules